# 2.1 · Data manipulation and tensors

**Machine Learning Foundations for AI Safety · Chapter 2**  
**Estimated time:** 30–40 minutes · **Prerequisite:** knowing what a Python list is.

### What you will be able to do

- Explain a tensor using everyday examples, identify its **shape** and **axes**, and tell a scalar, vector and matrix apart.
- Select a cell or a row/column using Python indexing.
- Explain the difference between *elementwise arithmetic* and *broadcasting*.
- Run small PyTorch examples and predict their outputs before running them.

> **Our running example:** A made-up class of four learners. Each learner has study hours, completed practice questions, and a quiz score. We use synthetic numbers for easy arithmetic; they are not evidence that hours alone determine a real student's score.

**Learning method:** Read the idea → inspect the picture → predict the output → run the code → explain it in your own words.


## 1. What does “data” look like to a computer?

We may write **“Ben studied 2 hours, completed 4 questions, and scored 5.”** A computer can process this observation as three numbers: `[2, 4, 5]`. Put four learners together and we get a rectangular **table**.

A **row** describes *one learner*. A **column** describes *one type of measurement*. A **cell** is where a particular row meets a particular column; every cell stores exactly one value.

Here is the entire table. Count the four learner rows and three numeric columns. The table therefore has shape **(4, 3)**. The names help humans read the data, but the names are not included in this particular numeric matrix.


<picture>
  <source type="image/svg+xml" srcset="images/01-table-rows-columns.svg">
  <img src="images/01-table-rows-columns.png" alt="A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted" loading="lazy">
</picture>

*Figure: A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted.*


For example, **row Ben + column Score = 5**. The table is not just a picture of cells: it is an organized way to tell the computer which value belongs where.

### Try this in Python

First build the numeric table with PyTorch, a library used widely in deep learning. We will not need a GPU or any downloaded dataset.


In [ ]:
import torch

learner_names = ["Alice", "Ben", "Chika", "Dara"]
data = torch.tensor([
    [1., 2., 3.],  # Alice: hours, practice, score
    [2., 4., 5.],  # Ben
    [3., 6., 7.],  # Chika
    [4., 8., 9.],  # Dara
])

print('Shape:', tuple(data.shape))
print('Number of values:', data.numel())
print('First row:', data[0].tolist())
assert tuple(data.shape) == (4, 3)


**Read the result:** A shape `(4, 3)` means 4 rows and 3 columns; `numel()` returns **12** because 4 × 3 = 12. The individual data values have not changed—only the way we grouped them determines the shape.

## 2. What is a tensor?

A **tensor** is an array of values organized along one or more **axes** (directions). A scalar is a single number; a vector is a one-dimensional array; a matrix is a two-dimensional array. In PyTorch, all three are tensors.

- **Scalar:** one quiz score, `5`; shape `()` (zero axes).
- **Vector:** four scores, `[3, 5, 7, 9]`; shape `(4,)` (one axis).
- **Matrix:** all the numerical observations; shape `(4, 3)` (two axes).
- **Higher-dimensional tensor:** a group of color images can be organized as `(number_of_images, color_channels, height, width)`.

Think of an **axis** as a direction you can move through the data: down the rows (axis 0) or across the columns (axis 1).


<picture>
  <source type="image/svg+xml" srcset="images/02-tensor-shapes.svg">
  <img src="images/02-tensor-shapes.png" alt="A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown" loading="lazy">
</picture>

*Figure: A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown.*


In [ ]:
scalar = torch.tensor(5.)
vector = torch.tensor([3., 5., 7., 9.])
matrix = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

for label, tensor in [('scalar', scalar), ('vector', vector), ('matrix', matrix)]:
    print(f'{label:6}: shape={tuple(tensor.shape)}, dimensions={tensor.ndim}')

# Reshaping changes the arrangement, not the individual values.
sequence = torch.arange(12)
print('Reshaped to 3 rows × 4 columns:\n', sequence.reshape(3, 4))


**Predict before running:** `torch.arange(12)` contains 12 numbers, starting at 0 and ending at 11. If we reshape it to `(3, 4)`, we get 3 rows of 4 numbers—not 12 new numbers. A reshape only works when the total number of entries is preserved.

## 3. Finding a particular value: indexing and slicing

Python starts counting positions at **0**, not 1. `data[1, 2]` means **second row, third column**. That is Ben’s score, `5`.

A colon (`:`) says “take all entries along this axis.” So `data[:, 0]` returns **the first column for every learner**. A slice like `data[0:2]` returns rows at positions 0 and 1, stopping *before* 2.


<picture>
  <source type="image/svg+xml" srcset="images/03-indexing.svg">
  <img src="images/03-indexing.png" alt="A rectangular tensor with explicit zero-based row and column indices and one highlighted entry" loading="lazy">
</picture>

*Figure: A rectangular tensor with explicit zero-based row and column indices and one highlighted entry.*


In [ ]:
print('One cell:', data[1, 2].item())
print('All study hours:', data[:, 0].tolist())
print('Ben’s entire row:', data[1].tolist())
print('First two learners:\n', data[0:2])

assert data[1, 2].item() == 5
assert data[:, 0].tolist() == [1., 2., 3., 4.]


## 4. Operations: elementwise vs. broadcasting

When two tensors have **the same shape**, an operation like `+` or `*` usually acts on **matching cells**. For example, `[1, 2] + [10, 20]` gives `[11, 22]`.

But what if you have a **column** of 3 numbers and a **row** of 2 numbers? PyTorch can often reuse entries along an axis of length 1. This is called **broadcasting**.

Read the picture **row by row**. For example, the top-right result cell is `1 + 20 = 21`; the bottom-left result cell is `3 + 10 = 13`. There are 3 rows and 2 columns in the result; no cell is left unspecified.


<picture>
  <source type="image/svg+xml" srcset="images/04-broadcasting.svg">
  <img src="images/04-broadcasting.png" alt="A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table" loading="lazy">
</picture>

*Figure: A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table.*


In [ ]:
A = torch.tensor([[1.], [2.], [3.]])   # three rows, one column
B = torch.tensor([[10., 20.]])           # one row, two columns

result = A + B
print('A shape:', tuple(A.shape))
print('B shape:', tuple(B.shape))
print('Result shape:', tuple(result.shape))
print(result)

assert result.tolist() == [[11., 21.], [12., 22.], [13., 23.]]


**Important distinction:** Broadcasting does **not** mean any two arbitrary shapes are compatible. For aligned axes, the lengths must match **or one of them must be 1**. For example, `(3, 2)` + `(3, 2)` works; `(3, 1)` + `(1, 2)` works; `(3, 2)` + `(4, 2)` does *not* work.

Also, `A * B` is **elementwise multiplication** (with broadcasting when applicable), while `A @ B` is **matrix multiplication**, a different operation we will study in Lesson 2.3.

## 5. Why this matters for machine learning

A learning model processes many examples at once. Rather than looping through every learner and every feature, tensor operations can apply calculations efficiently to the entire data structure. We will use these same ideas later for neural-network inputs, weights and activations.

### Mini-experiment: change just one number


In [ ]:
experiment = data.clone()  # we create a copy, leaving the original unchanged
experiment[1, 2] = 6.
print('Old Ben score:', data[1, 2].item())
print('Edited Ben score:', experiment[1, 2].item())
print('The edited copy differs from the original:', not torch.equal(data, experiment))  # True


### Check your understanding (answer before revealing by running)

1. A tensor has shape `(5, 4)`. How many **rows**, **columns**, and **values** are there?
2. Which expression selects Chika’s **hours** from `data`?
3. What is the result shape of `(2, 1)` + `(1, 3)`? What are the result’s dimensions?
4. Explain **in one sentence** why `*` is not the same as `@`.

**Practice cell:** Try `data[2, 0]`, `data[:, 1]`, and a broadcast of `[[5], [6]] + [[0, 10, 20]]`.

**Takeaway:** The value of a number and the *position* it occupies in a tensor are different ideas. Understanding shape and indexing prevents many later ML errors.

➡️ **Next:** [2.2 Data preprocessing](02-data-preprocessing.ipynb) — real tables often include missing values and categories.


In [ ]:
# Your turn — edit these expressions and predict their outputs.
print('Chika’s hours:', data[2, 0].item())
print('All practice values:', data[:, 1].tolist())
print('Broadcast example:\n', torch.tensor([[5], [6]]) + torch.tensor([[0, 10, 20]]))
